In [1]:
import os
import calendar
import requests
import numpy as np
import pandas as pd
import xarray as xr


YEAR = 2025
VARIABLE = "rainfall"
RESOLUTION = "25km"
FREQUENCY = "day"

OUT_DIR = "./metoffice_hadukgrid_2025_all_locations"
FINAL_CSV = os.path.join(
    OUT_DIR,
    f"hadukgrid_{VARIABLE}_{RESOLUTION}_{YEAR}_ml_ready_final.csv"
)

os.makedirs(OUT_DIR, exist_ok=True)


def last_day_of_month(year, month):
    return calendar.monthrange(year, month)[1]

def build_nc_url(year, month):
    yyyymm = f"{year}{month:02d}"
    start = f"{yyyymm}01"
    end = f"{yyyymm}{last_day_of_month(year, month):02d}"
    return (
        f"https://www.metoffice.gov.uk/hadobs/hadukgrid/data/{year}/"
        f"{VARIABLE}_hadukgrid_uk_{RESOLUTION}_{FREQUENCY}_{start}-{end}.nc"
    )

def download_file(url, out_path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)


monthly_dfs = []

for month in range(1, 13):
    print(f"Processing {YEAR}-{month:02d}")
    url = build_nc_url(YEAR, month)
    nc_path = os.path.join(OUT_DIR, f"{YEAR}_{month:02d}.nc")

    download_file(url, nc_path)

    ds = xr.open_dataset(nc_path)
    data_var = list(ds.data_vars)[0]

    df = (
        ds[[data_var]]
        .to_dataframe()
        .reset_index()
        .rename(columns={data_var: "rainfall_mm"})
    )

    monthly_dfs.append(df)
    ds.close()

data = pd.concat(monthly_dfs, ignore_index=True)



data["time"] = pd.to_datetime(data["time"])
data = data.sort_values(["latitude", "longitude", "time"]).reset_index(drop=True)

data["day_of_year"] = data["time"].dt.dayofyear
data["month"] = data["time"].dt.month
data["week_of_year"] = data["time"].dt.isocalendar().week.astype(int)


data["season"] = np.select(
    [
        data["month"].isin([12, 1, 2]),
        data["month"].isin([3, 4, 5]),
        data["month"].isin([6, 7, 8]),
        data["month"].isin([9, 10, 11]),
    ],
    ["Winter", "Spring", "Summer", "Autumn"],
    default="Unknown"
)


group_cols = ["latitude", "longitude"]

for lag in [1, 3, 7]:
    data[f"rain_lag_{lag}"] = (
        data.groupby(group_cols)["rainfall_mm"].shift(lag)
    )

data["rain_rollsum_7"] = (
    data.groupby(group_cols)["rainfall_mm"]
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=[0, 1], drop=True)
)

data["rain_rollsum_30"] = (
    data.groupby(group_cols)["rainfall_mm"]
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=[0, 1], drop=True)
)


data = data.dropna().reset_index(drop=True)

data.to_csv(FINAL_CSV, index=False)

print("\n==============================")
print(" DATASET READY")
print("Saved to:", FINAL_CSV)
print("Rows:", len(data))
print("Columns:", list(data.columns))
print("==============================\n")

print(data.head(10))


Processing 2025-01
Processing 2025-02
Processing 2025-03
Processing 2025-04
Processing 2025-05
Processing 2025-06
Processing 2025-07
Processing 2025-08
Processing 2025-09
Processing 2025-10
Processing 2025-11
Processing 2025-12

 DATASET READY
Saved to: ./metoffice_hadukgrid_2025_all_locations\hadukgrid_rainfall_25km_2025_ml_ready_final.csv
Rows: 156088
Columns: ['time', 'projection_y_coordinate', 'projection_x_coordinate', 'rainfall_mm', 'latitude', 'longitude', 'day_of_year', 'month', 'week_of_year', 'season', 'rain_lag_1', 'rain_lag_3', 'rain_lag_7', 'rain_rollsum_7', 'rain_rollsum_30']

                 time  projection_y_coordinate  projection_x_coordinate  \
0 2025-01-08 12:00:00                  37500.0                 162500.0   
1 2025-01-09 12:00:00                  37500.0                 162500.0   
2 2025-01-10 12:00:00                  37500.0                 162500.0   
3 2025-01-11 12:00:00                  37500.0                 162500.0   
4 2025-01-12 12:00:00      